In [ ]:
import os
from src import RASPRoutines
from src import AnalysisFunctions
from src import PlottingFunctions
from src import IOFunctions
import numpy as np
import matplotlib.pyplot as plt

IO = IOFunctions.IO_Functions()
RASP = RASPRoutines.RASP_Routines()
A_F = AnalysisFunctions.Analysis_Functions()
plotter = PlottingFunctions.Plotter()

In [ ]:
""" the commands below should make a comparison plot between the cell mask of
    interest with no thresholding and with some thresholding strategy, using
    threshold_cell_areas_3d (the default 3D cell-size/shape filter -- removes
    small/large objects, over-filled planes, and objects that don't span enough
    z-planes, then closes the mask with a small dilation/erosion).
"""

cell_mask = IO.read_tiff(
    os.path.abspath(r"example_images_analysis/Example_ProteinImage_C0_cellMask.tiff")
)  # example file, replace as you wish. Always input as the absolute path of a raw string
# This is the raw *_cellMask.tiff written by analyse_images -- not yet size-filtered.

lower_size_threshold = 1000.0  # lower size threshold (voxels)
upper_size_threshold = (
    np.inf
)  # upper size threshold. Put as np.inf/do not put in for no upper size threshold.

cell_mask_new, pil, areas, centroids = A_F.threshold_cell_areas_3d(
    cell_mask,
    lower_cell_size_threshold=lower_size_threshold,
    upper_cell_size_threshold=upper_size_threshold,
)
# cell_mask/cell_mask_new come back as (z, height, width) -- z is axis 0, not
# the last axis -- so project over axis=0 for a 2D comparison plot below.
cell_mask_old = np.sum(cell_mask, axis=0).clip(0, 1)
cell_mask_new_2d = np.sum(cell_mask_new, axis=0).clip(0, 1)

In [ ]:
fig, axs = plotter.two_column_plot(ncolumns=2, widthratio=[1, 1])

axs[0] = plotter.image_plot(axs=axs[0], data=cell_mask_old, cbar="off")
axs[1] = plotter.image_plot(axs=axs[1], data=cell_mask_new_2d, cbar="off")

axs[0].set_title("Mask with no thresholding")
axs[1].set_title("Mask with size thresholding")
plt.show(block=False)

In [ ]:
image_size = cell_mask_new.shape  # (z, height, width) of the 3D mask
for k in np.arange(len(areas)):
    coords = pil[k]
    zm, xm, ym = coords[:, 0], coords[:, 1], coords[:, 2]
    if (
        np.any(xm > image_size[1])
        or np.any(ym > image_size[2])
        or np.any(zm > image_size[0])
    ):
        print(k)